In [1]:
import pandas as pd
import numpy as np

# Загрузка данных о заказах
orders_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv"
]

df_orders = pd.concat(
    [pd.read_csv(f) for f in orders_files],
    ignore_index=True
)

print("Исходные данные:")
print(df_orders.head())

# Проверка структуры данных
print("Информация о данных:")
print(df_orders.info())

# Средняя цена позиции для каждого пользователя
avg_price = (
    df_orders
    .groupby("user_id")["price"]
    .mean()
    .rename("avg_price")
)

# Среднее количество товаров в одном заказе
order_quantities = (
    df_orders
    .groupby(["user_id", "order_id"])["quantity"]
    .sum()
    .reset_index(name="order_quantity")
)

avg_quantity = (
    order_quantities
    .groupby("user_id")["order_quantity"]
    .mean()
    .rename("avg_quantity")
)

# Объединение признаков пользователей
user_stats = pd.concat(
    [avg_price, avg_quantity],
    axis=1
)

print("Средняя цена и среднее количество товаров:")
print(user_stats.head())

# Z-нормализация признаков
user_stats["avg_price_z"] = (
    (user_stats["avg_price"] - user_stats["avg_price"].mean())
    / user_stats["avg_price"].std(ddof=0)
)

user_stats["avg_quantity_z"] = (
    (user_stats["avg_quantity"] - user_stats["avg_quantity"].mean())
    / user_stats["avg_quantity"].std(ddof=0)
)

print("Данные после Z-нормализации:")
print(user_stats.head())

# Пользователь с максимальным Z-значением средней цены
max_user = user_stats["avg_price_z"].idxmax()

print("Пользователь с максимальным Z-значением средней цены:")
print(user_stats.loc[max_user])

# Пользователь с минимальным Z-значением средней цены
min_user = user_stats["avg_price_z"].idxmin()

print("Пользователь с минимальным Z-значением средней цены:")
print(user_stats.loc[min_user])

# Сохранение результатов
user_stats.to_csv(
    "TP5_Variant9_results.csv",
    encoding="utf-8-sig"
)

print("Результаты TP5 успешно сохранены.")

Исходные данные:
   order_id  user_id       order_datetime  product_id     price  quantity  \
0         1     1044  2024-01-03 18:39:26          26  14038.15         1   
1         2     1079  2024-01-03 17:45:43          16   9172.60         1   
2         3     1041  2024-01-06 22:46:38          25   9035.26         2   
3         4     1083  2024-01-08 21:03:51           9  15234.33         3   
4         5     1068  2024-01-03 23:26:53          24   4134.38         2   

     status  
0  refunded  
1      paid  
2      paid  
3      paid  
4      paid  
Информация о данных:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        15000 non-null  int64  
 1   user_id         15000 non-null  int64  
 2   order_datetime  15000 non-null  str    
 3   product_id      15000 non-null  int64  
 4   price           15000 non-null  float64
 5 